# 🧪 Expérimentation des Méthodes de Chunking pour RAG

## Objectif
Trouver la meilleure méthode de découpage de documents pour le système RAG du ministère.
    
## Méthodes à Tester
1. **Fixed Chunk** - Découpage fixe (basique)
2. **Sentence Chunking** - Par phrases complètes
3. **Paragraph Chunking** - Par paragraphes
4. **Semantic Chunking** - Par changement de sujet
5. **Hierarchical Chunking** - Hiérarchie documentaire
    
## Métriques d'Évaluation
    - Cohérence sémantique
    - Pertinence des réponses
    - Taille moyenne des chunks
    - Temps de traitement
    - Qualité de la récupération

In [1]:


with open("..\\data\\infos.md", "r", encoding="utf-8") as f:
        texte_complet = f.read()

In [2]:

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)
# Pour le Semantic Chunking (alternative simple sans API externe via LangChain)
#from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.embeddings import FakeEmbeddings # Remplacer par OpenAIEmbeddings ou HuggingFaceEmbeddings en prod

# ---------------------------------------------------------
# 0. Ton document de test (Exemple typique)
# ---------------------------------------------------------

document_text = texte_complet
print(f"Longueur du document original : {len(document_text)} caractères.\n")


# ---------------------------------------------------------
# 1. Fixed Chunking (Découpage fixe)
# ---------------------------------------------------------
print("--- 1. FIXED CHUNKING ---")
fixed_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=200,
    chunk_overlap=30
)
fixed_chunks = fixed_splitter.create_documents([document_text])
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} cars):\n{chunk.page_content}\n{'-'*20}")


# ---------------------------------------------------------
# 2. Sentence / Recursive Chunking (Découpage par phrase/mot)
# ---------------------------------------------------------
print("\n--- 2. RECURSIVE / SENTENCE CHUNKING ---")
# On utilise des séparateurs logiques (paragraphe, puis phrase, puis mot) pour ne pas couper au milieu d'une phrase
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " ", ""],
    chunk_size=250,
    chunk_overlap=20
)
recursive_chunks = recursive_splitter.create_documents([document_text])
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} cars):\n{chunk.page_content}\n{'-'*20}")


# ---------------------------------------------------------
# 3. Paragraph Chunking (Découpage par paragraphe)
# ---------------------------------------------------------
print("\n--- 3. PARAGRAPH CHUNKING ---")
# On force la coupe uniquement sur les doubles sauts de ligne
paragraph_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=0, # On désactive la contrainte de taille pour forcer le séparateur
    chunk_overlap=0
)
paragraph_chunks = paragraph_splitter.create_documents([document_text])
for i, chunk in enumerate(paragraph_chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} cars):\n{chunk.page_content}\n{'-'*20}")


# ---------------------------------------------------------
# 4. Semantic Chunking (Découpage sémantique)
# ---------------------------------------------------------
print("\n--- 4. SEMANTIC CHUNKING ---")
# Note : En production, utilise un vrai modèle d'embedding (ex: OpenAIEmbeddings() ou HuggingFaceEmbeddings())
# Ici on utilise FakeEmbeddings pour que le code tourne sans clé API, basé sur la taille des phrases.
embeddings = FakeEmbeddings(size=1536)
semantic_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" # Coupe quand la distance sémantique dépasse un certain percentile
)
semantic_chunks = semantic_splitter.create_documents([document_text])
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n{'-'*20}")


# ---------------------------------------------------------
# 5. Hierarchical Chunking (Markdown / Parent-Child)
# ---------------------------------------------------------
print("\n--- 5. HIERARCHICAL CHUNKING (Markdown) ---")
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(document_text)

for i, chunk in enumerate(md_header_splits):
    print(f"Chunk {i+1} - Métadonnées: {chunk.metadata}")
    print(f"Contenu:\n{chunk.page_content}\n{'-'*20}")

e:\cours ifri\Programmation et BD\Python\Projet IA\DTDI_Managment\SIOU\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Longueur du document original : 25868 caractères.

--- 1. FIXED CHUNKING ---
Chunk 1 (200 cars):
# Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin

## Direction de la Digitalisation

### Cadre Légal et Création
La Direction de la Digitalisation (DD
--------------------
Chunk 2 (200 cars):
ction de la Digitalisation (DD) a été créée par **arrêté n°011/MND/DC/SGM/CTJ/DAF/CJ/SA/024SGG du 27 août 2020** portant attributions, organisation et fonctionnement des directions techniques du Minis
--------------------
Chunk 3 (200 cars):
directions techniques du Ministère du Numérique et de la Digitalisation.

Passation de service effectuée le **lundi 16 novembre 2020** entre Monsieur Houégnon Geoffroy BONOU (sortant) et Monsieur Bori
--------------------
Chunk 4 (200 cars):
NOU (sortant) et Monsieur Boris Rodrigue SEHLOUAN (entrant), sous la supervision de Monsieur Eugène ALLEY, Inspecteur Général du Ministère.

### Missions et Attributions
La dire

ValueError: chunk_size must be > 0, got 0

In [ ]:
import os

os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

# Semantic embedding avec sentences transformers et le model BAAI/bge-m3

from huggingface_hub import snapshot_download

print("Téléchargement des fichiers en cours...")

snapshot_download(repo_id="BAAI/bge-m3" , ignore_patterns=["*.jpg", "*.png", "*.md", "imgs/*", "README.md"])

from langchain_huggingface import HuggingFaceEmbeddings

# Initialisation du modèle BGE-M3 (excellent pour le français)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)


text = "LangChain facilite l'intégration des embeddings sémantiques."
vector = embeddings.embed_query(text)


Téléchargement des fichiers en cours...


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]e:\cours ifri\Programmation et BD\Python\Projet IA\DTDI_Managment\SIOU\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Freud Arthur\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


: 

: 

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

print("🧠 Chargement du modèle paraphrase depuis le dossier local...")

# Configuration pointant directement vers ton dossier local
embeddings = HuggingFaceEmbeddings(
    model_name="../pretrained/paraphrase-local",
    model_kwargs={'device': 'cpu'},  # Utilise 'cuda' si tu as un GPU disponible
    encode_kwargs={'normalize_embeddings': True}
)

# Test rapide pour valider que tout fonctionne hors-ligne
text = "Vérification de la configuration locale pour SIOu."
vector = embeddings.embed_query(text)

print(f"🚀 Modèle chargé instantanément sans internet !")
print(f"Dim des vecteurs : {len(vector)} (Attendu : 384)")

e:\cours ifri\Programmation et BD\Python\Projet IA\DTDI_Managment\SIOU\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧠 Chargement du modèle paraphrase depuis le dossier local...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2796.38it/s]


🚀 Modèle chargé instantanément sans internet !
Dim des vecteurs : 384 (Attendu : 384)


In [5]:
# ---------------------------------------------------------
# TESTING SPECIFIC SPLITTERS AS REQUESTED
# ---------------------------------------------------------
import time
from typing import List, Dict, Any

def evaluate_chunks(chunks: List[Any], method_name: str) -> Dict[str, Any]:
    """Evaluate the quality of chunks using various metrics"""
    
    # Basic metrics
    num_chunks = len(chunks)
    avg_length = sum(len(chunk.page_content) for chunk in chunks) / num_chunks if num_chunks > 0 else 0
    min_length = min(len(chunk.page_content) for chunk in chunks) if num_chunks > 0 else 0
    max_length = max(len(chunk.page_content) for chunk in chunks) if num_chunks > 0 else 0
    
    # Semantic coherence (simple heuristic - count complete sentences)
    complete_sentences = 0
    for chunk in chunks:
        content = chunk.page_content
        # Count sentences ending with punctuation
        sentences = [s for s in content.split('.') if len(s.strip()) > 10]  # Filter out very short "sentences"
        complete_sentences += len(sentences)
    
    # Header preservation (for markdown)
    headers_preserved = 0
    for chunk in chunks:
        content = chunk.page_content
        if any(header in content for header in ['# ', '## ', '### ']):
            headers_preserved += 1
    
    return {
        'method': method_name,
        'num_chunks': num_chunks,
        'avg_length': round(avg_length, 2),
        'min_length': min_length,
        'max_length': max_length,
        'complete_sentences': complete_sentences,
        'sentences_per_chunk': round(complete_sentences / num_chunks, 2) if num_chunks > 0 else 0,
        'headers_preserved': headers_preserved,
        'header_preservation_rate': round(headers_preserved / num_chunks, 2) if num_chunks > 0 else 0
    }

def print_evaluation(results: Dict[str, Any]):
    """Print evaluation results in a readable format"""
    print(f"\n📊 ÉVALUATION: {results['method']}")
    print(f"Nombre de chunks: {results['num_chunks']}")
    print(f"Taille moyenne: {results['avg_length']} caractères")
    print(f"Taille min/max: {results['min_length']}-{results['max_length']} caractères")
    print(f"Phrases complètes par chunk: {results['sentences_per_chunk']}")
    print(f"Taux de préservation des en-têtes: {results['header_preservation_rate']}")


In [6]:
# ---------------------------------------------------------
# 1. MARKDOWN SPLITTER TESTING (as requested)
# ---------------------------------------------------------
print("\n" + "="*60)
print("🔍 TEST 1: MARKDOWN SPLITTER")
print("="*60)

from langchain_text_splitters import MarkdownHeaderTextSplitter

# Configuration pour capturer la structure markdown
headers_to_split_on = [
    ("#", "Section"),
    ("##", "Sous-section"), 
    ("###", "Paragraphe"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = markdown_splitter.split_text(document_text)

print(f"Markdown Splitter - Nombre de chunks: {len(md_chunks)}")
for i, chunk in enumerate(md_chunks[:3]):  # Show first 3 chunks as example
    print(f"\nChunk {i+1} - Type: {chunk.metadata.get('Header 1', 'Section')}")
    print(f"Contenu (premier 100 chars): {chunk.page_content[:100]}...")

# Evaluation
md_results = evaluate_chunks(md_chunks, "Markdown Splitter")
print_evaluation(md_results)



🔍 TEST 1: MARKDOWN SPLITTER
Markdown Splitter - Nombre de chunks: 29

Chunk 1 - Type: Section
Contenu (premier 100 chars): La Direction de la Digitalisation (DD) a été créée par **arrêté n°011/MND/DC/SGM/CTJ/DAF/CJ/SA/024SG...

Chunk 2 - Type: Section
Contenu (premier 100 chars): La direction de la Digitalisation est chargée de :
- superviser la mise en œuvre du programme de gou...

Chunk 3 - Type: Section
Contenu (premier 100 chars): Outre le secrétariat, elle est composée de deux (02) services :
- le **Service de l’Administration E...

📊 ÉVALUATION: Markdown Splitter
Nombre de chunks: 29
Taille moyenne: 839.07 caractères
Taille min/max: 44-3846 caractères
Phrases complètes par chunk: 2.31
Taux de préservation des en-têtes: 0.14


In [7]:
# ---------------------------------------------------------
# 2. RECURSIVE SPLITTER TESTING (as requested)
# ---------------------------------------------------------
print("\n" + "="*60)
print("🔍 TEST 2: RECURSIVE SPLITTER")
print("="*60)

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuration optimisée pour le français
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],  # Séparateurs logiques pour le français
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

recursive_chunks = recursive_splitter.create_documents([document_text])

print(f"Recursive Splitter - Nombre de chunks: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks[:3]):  # Show first 3 chunks as example
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"Contenu (premier 100 chars): {chunk.page_content[:100]}...")

# Evaluation
recursive_results = evaluate_chunks(recursive_chunks, "Recursive Splitter")
print_evaluation(recursive_results)



🔍 TEST 2: RECURSIVE SPLITTER
Recursive Splitter - Nombre de chunks: 128

Chunk 1 (133 chars):
Contenu (premier 100 chars): # Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin

...

Chunk 2 (277 chars):
Contenu (premier 100 chars): ### Cadre Légal et Création
La Direction de la Digitalisation (DD) a été créée par **arrêté n°011/MN...

Chunk 3 (235 chars):
Contenu (premier 100 chars): Passation de service effectuée le **lundi 16 novembre 2020** entre Monsieur Houégnon Geoffroy BONOU ...

📊 ÉVALUATION: Recursive Splitter
Nombre de chunks: 128
Taille moyenne: 202.69 caractères
Taille min/max: 17-299 caractères
Phrases complètes par chunk: 1.17
Taux de préservation des en-têtes: 0.34


In [8]:
# ---------------------------------------------------------
# 3. COMBINED APPROACH: MARKDOWN + RECURSIVE (as requested)
# ---------------------------------------------------------
print("\n" + "="*60)
print("🔍 TEST 3: COMBINED MARKDOWN + RECURSIVE SPLITTER")
print("="*60)

# D'abord, on découpe par structure markdown
markdown_splitter_combined = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_sections = markdown_splitter_combined.split_text(document_text)

# Ensuite, on applique le recursive splitter sur chaque section
combined_chunks = []
for section in md_sections:
    section_text = section.page_content
    # Appliquer le recursive splitter sur chaque section markdown
    section_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        chunk_size=250,
        chunk_overlap=30,
        length_function=len,
        is_separator_regex=False,
    )
    section_chunks = section_splitter.create_documents([section_text])
    
    # Conserver les métadonnées markdown
    for chunk in section_chunks:
        chunk.metadata.update(section.metadata)
        combined_chunks.append(chunk)

print(f"Combined Splitter - Nombre de chunks: {len(combined_chunks)}")
for i, chunk in enumerate(combined_chunks[:3]):  # Show first 3 chunks as example
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"Métadonnées: {chunk.metadata}")
    print(f"Contenu (premier 100 chars): {chunk.page_content[:100]}...")

# Evaluation
combined_results = evaluate_chunks(combined_chunks, "Combined Markdown+Recursive Splitter")
print_evaluation(combined_results)



🔍 TEST 3: COMBINED MARKDOWN + RECURSIVE SPLITTER
Combined Splitter - Nombre de chunks: 141

Chunk 1 (248 chars):
Métadonnées: {'Section': 'Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin', 'Sous-section': 'Direction de la Digitalisation', 'Paragraphe': 'Cadre Légal et Création'}
Contenu (premier 100 chars): La Direction de la Digitalisation (DD) a été créée par **arrêté n°011/MND/DC/SGM/CTJ/DAF/CJ/SA/024SG...

Chunk 2 (1 chars):
Métadonnées: {'Section': 'Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin', 'Sous-section': 'Direction de la Digitalisation', 'Paragraphe': 'Cadre Légal et Création'}
Contenu (premier 100 chars): ....

Chunk 3 (235 chars):
Métadonnées: {'Section': 'Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin', 'Sous-section': 'Direction de la Digitalisation', 'Paragraphe': 'Cadre Légal et Création'}
Contenu (premier 100 chars): 

In [10]:
# ---------------------------------------------------------
# 4. COMPARATIVE ANALYSIS AND RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*60)
print("📈 ANALYSE COMPARATIVE")
print("="*60)

all_results = [md_results, recursive_results, combined_results]

# Create comparison table
print("\n{:<30} {:<15} {:<15} {:<20} {:<25}".format("Méthode", "Chunks", "Taille moy", "Phrases/chunk", "Préservation en-têtes"))
print("-" * 100)

for result in all_results:
    print("\n{:<30} {:<15} {:<15} {:<20} {:<25}".format(
        result['method'],
        result['num_chunks'],
        result['avg_length'],
        result['sentences_per_chunk'],
        f"{result['header_preservation_rate']*100}%"
    ))

# Recommendations based on document type
print("\n💡 RECOMMANDATIONS:")
print("• Pour les documents structurés (markdown, rapports): La méthode combinée offre le meilleur équilibre")
print("• Pour les textes courts ou non structurés: Le Recursive Splitter est plus simple et efficace")
print("• Pour préserver la structure documentaire: Le Markdown Splitter seul est idéal")
print("\n✅ Test terminé! Les trois méthodes ont été évaluées avec des métriques objectives.")



📈 ANALYSE COMPARATIVE

Méthode                        Chunks          Taille moy      Phrases/chunk        Préservation en-têtes    
----------------------------------------------------------------------------------------------------

Markdown Splitter              29              839.07          2.31                 14.000000000000002%      

Recursive Splitter             128             202.69          1.17                 34.0%                    

Combined Markdown+Recursive Splitter 141             174.49          1.18                 10.0%                    

💡 RECOMMANDATIONS:
• Pour les documents structurés (markdown, rapports): La méthode combinée offre le meilleur équilibre
• Pour les textes courts ou non structurés: Le Recursive Splitter est plus simple et efficace
• Pour préserver la structure documentaire: Le Markdown Splitter seul est idéal

✅ Test terminé! Les trois méthodes ont été évaluées avec des métriques objectives.


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

print("🧠 Chargement du modèle paraphrase depuis le dossier local...")

# Configuration pointant directement vers ton dossier local
embeddings = HuggingFaceEmbeddings(
    model_name="../pretrained/paraphrase-local",
    model_kwargs={'device': 'cpu'},  # Utilise 'cuda' si tu as un GPU disponible
    encode_kwargs={'normalize_embeddings': True}
)

# Test rapide pour valider que tout fonctionne hors-ligne
text = "Vérification de la configuration locale pour SIOu."
vector = embeddings.embed_query(text)

print(f"🚀 Modèle chargé instantanément sans internet !")
print(f"Dim des vecteurs : {len(vector)} (Attendu : 384)")

with open("..\\data\\infos.md", "r", encoding="utf-8") as f:
        texte_complet = f.read()

🧠 Chargement du modèle paraphrase depuis le dossier local...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6290.35it/s]


🚀 Modèle chargé instantanément sans internet !
Dim des vecteurs : 384 (Attendu : 384)


In [13]:
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Recherche sémantique (vectorielle)
def split_administrative_document(
    document_text: str, 
    headers_to_split_on: List[Tuple[str, str]] = None, # type: ignore
    chunk_size: int = 600,
    chunk_overlap: int = 100
) -> List[Document]:
    """
    Découpe un document administratif (Décret, AOF) de manière hybride :
    1. Découpage structurel basé sur les balises Markdown.
    2. Découpage granulaire via un RecursiveCharacterTextSplitter sur chaque section.
    3. Préservation et propagation des métadonnées structurelles (titres, articles).
    
    Args:
        document_text (str): Le texte brut du document à découper.
        headers_to_split_on (List[Tuple[str, str]]): Les en-têtes Markdown à cibler 
            (ex: [("#", "Titre"), ("##", "Chapitre"), ("###", "Article")]).
        chunk_size (int): La taille maximale (en caractères) de chaque sous-chunk.
        chunk_overlap (int): Le chevauchement (en caractères) entre deux sous-chunks.
        
    Returns:
        List[Document]: Une liste d'objets Document LangChain prêts pour l'indexation.
    """
    # Configuration par défaut des en-têtes si non fournis
    if headers_to_split_on is None:
        headers_to_split_on = [
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ]
        
    # Étape 1 : Découpage par structure Markdown
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    md_sections = markdown_splitter.split_text(document_text)
    
    # Étape 2 : Initialisation du splitter récursif
    # Ajout des marqueurs spécifiques aux textes administratifs (ex: "\n\nArticle ", "\nArticle ")
    section_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\nArticle ", "\nArticle ", "\n\n", "\n", ". ", "! ", "? ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    
    combined_chunks = []
    
    # Étape 3 : Application récursive et fusion des métadonnées
    for section in md_sections:
        # Extraire les sous-chunks pour la section courante
        section_chunks = section_splitter.create_documents([section.page_content])
        
        for chunk in section_chunks:
            # On propage les métadonnées de la section Markdown parente
            chunk.metadata.update(section.metadata)
            combined_chunks.append(chunk)
            
    return combined_chunks

In [19]:
import numpy as np
chunks = split_administrative_document(texte_complet)
print(f"📦 Nombre de chunks générés : {len(chunks)}\n")

texts_to_embed = [doc.page_content for doc in chunks]

# 2. Vectorisation (Cette fois, ça va passer sans erreur !)
print("⏳ Vectorisation des chunks...")
chunks_embeddings = embeddings.embed_documents(texts_to_embed)

# 4. Ta requête utilisateur
query = "Qui s'occupe de la e-administration et de la dématérialisation ?"

print(f"🔍 Requête test : '{query}'")

# Vectorisation de la requête
query_embedding = embeddings.embed_query(query)

# 5. Calcul de la similarité (Recherche sémantique basique)
# On utilise le produit scalaire puisque nos embeddings sont normalisés
def cosine_similarity(v1, v2):
    return np.dot(v1, v2)

scores = [cosine_similarity(query_embedding, chunk_emb) for chunk_emb in chunks_embeddings]

# Trouver les indices des 3 meilleurs scores
top_k = 3
best_indices = np.argsort(scores)[::-1][:top_k]

print(f"\n--- 🎯 TOP {top_k} DES RÉSULTATS DU RETRIEVAL ---")
for rank, idx in enumerate(best_indices, 1):
    print(f"\n[Position #{rank}] - Score : {scores[idx]:.4f}")
    print(f"Métadonnées : {chunks[idx].metadata}")
    print(f"Contenu :\n{chunks[idx].page_content}")
    print("-" * 40)

📦 Nombre de chunks générés : 62

⏳ Vectorisation des chunks...
🔍 Requête test : 'Qui s'occupe de la e-administration et de la dématérialisation ?'

--- 🎯 TOP 3 DES RÉSULTATS DU RETRIEVAL ---

[Position #1] - Score : 0.6250
Métadonnées : {'Header_1': 'Base de Connaissances : Attributions et Fonctionnement des Entités du Secteur Numérique au Bénin', 'Header_2': 'Direction de la Digitalisation', 'Header_3': 'Missions et Attributions'}
Contenu :
La direction de la Digitalisation est chargée de :
- superviser la mise en œuvre du programme de gouvernance électronique de l’État par l’usage des Technologies de l’Information et de la Communication dans l’Administration et la dématérialisation des services publics ;
- promouvoir les plateformes digitales et les solutions numériques ;
- s’assurer de la vulgarisation du schéma directeur national des systèmes d’information et de sa mise en œuvre ;
- contribuer au développement de l’entrepreneuriat numérique ;
- Traitement des demandes d’audience.
-

In [23]:
import numpy as np
from rank_bm25 import BM25Okapi

# --- CONFIGURATION INITIALE ---
query = "Qui a le pouvoir de nommer un Chef de Division ?"
texts_to_embed = [doc.page_content for doc in chunks]

print(f"🔍 Expérimentation Hybride pour la requête : '{query}'\n")


query_embedding = embeddings.embed_query(query)
dense_scores = [np.dot(query_embedding, chunk_emb) for chunk_emb in chunks_embeddings]

# On classe les chunks du meilleur au moins bon pour le Dense
dense_ranking = np.argsort(dense_scores)[::-1]


# ==========================================
# 2. BRANCHE MOTS-CLÉS (SPARSE - BM25)
# ==========================================
# Le BM25 travaille sur les mots (tokens). On nettoie et découpe grossièrement.
tokenized_corpus = [doc.lower().split(" ") for doc in texts_to_embed]
tokenized_query = query.lower().split(" ")

bm25 = BM25Okapi(tokenized_corpus)
sparse_scores = bm25.get_scores(tokenized_query)

# On classe les chunks du meilleur au moins bon pour le Sparse
sparse_ranking = np.argsort(sparse_scores)[::-1]


# ==========================================
# 3. FUSION DES RANGS (RRF - Reciprocal Rank Fusion)
# ==========================================
# RRF donne un score basé sur la POSITION du chunk dans les deux classements.
# Si un chunk est #1 en BM25 et #3 en Sémantique, son score RRF sera très élevé.
rrf_scores = np.zeros(len(chunks))
k = 60 # Paramètre standard de l'algorithme RRF

for rank, idx in enumerate(dense_ranking):
    rrf_scores[idx] += 1.0 / (k + (rank + 1))

for rank, idx in enumerate(sparse_ranking):
    rrf_scores[idx] += 1.0 / (k + (rank + 1))

# Classement final hybride
hybrid_ranking = np.argsort(rrf_scores)[::-1]


# ==========================================
# 4. AFFICHAGE DES RÉSULTATS COMPARATIFS
# ==========================================
print("=== 🧪 COMPARAISON DES POSITIONNEMENTS ===")
# On cherche où se trouve le premier morceau qui parle VRAIMENT de la "Direction du Numérique"
# Affichons le Top 3 Hybride pour analyser l'impact

for position, idx in enumerate(hybrid_ranking[:3], 1):
    print(f"\n[Position Hybride #{position}]")
    print(f"-> Rang initial Sémantique : #{list(dense_ranking).index(idx) + 1}")
    print(f"-> Rang initial BM25 (Mots-clés) : #{list(sparse_ranking).index(idx) + 1}")
    print(f"Score RRF Fusionné : {rrf_scores[idx]:.5f}")
    print(f"Métadonnées : {chunks[idx].metadata}")
    print(f"Contenu :\n{chunks[idx].page_content}")
    print("-" * 50)

🔍 Expérimentation Hybride pour la requête : 'Qui a le pouvoir de nommer un Chef de Division ?'

=== 🧪 COMPARAISON DES POSITIONNEMENTS ===

[Position Hybride #1]
-> Rang initial Sémantique : #1
-> Rang initial BM25 (Mots-clés) : #1
Score RRF Fusionné : 0.03279
Métadonnées : {'Header_1': 'Ministère du Numérique et de la Digitalisation - Base de Connaissances', 'Header_2': 'Arrêté N° 011/MIND/DG/SGMP/CGJ/DAF/CJ/SA/024SGG', 'Header_3': 'Chapitre Quatrième : Dispositions Finales'}
Contenu :
**Article 11 : Création des divisions et modalités de nomination de leurs Chefs**
Les services sont structurés en divisions.
Une note de service du Directeur technique précise les attributions des divisions.
Le Chef de Division est nommé par note de service du Directeur technique, sur proposition du Chef de service dont il relève.  
**Article 12 : Application**
Le Secrétaire Général du Ministère et les Directeurs techniques sont chargés, chacun en ce qui le concerne, de l’application du présent arrêté.  